This notebook runs the network intrusion detection system.

- Load the trained model  
- Collects live data for 20 seconds   
- Every 20 seconds, checks if traffic is normal or suspicious  
- If an attack is detected, shows a warning and sends a popup notification

In [ ]:
import joblib
from notifypy import Notify
import time
import pandas as pd
import sys
import os
from colorama import Fore, Style

In [ ]:
# Load the model
loaded_model = joblib.load(filename="IsolationForest_2026_03_23_205410.pkl")

In [ ]:
def popup_notification(title: str, msg: str):
    notification = Notify()
    notification.title = title
    notification.message = msg
    notification.icon = "NIDS-icon.png"

    notification.send()

In [ ]:
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("__file__"), "../..")))

from src.monitoring.helper import collect_monitored_data
from src.training.helper import aggregate_dataframe_into_seconds_interval


SECONDS_TO_PREDICT = 20
PREDICTION_THRESHOLD = 0.5

data_buffer = []
previous_network_io = None

is_supervised = hasattr(loaded_model, "predict_proba")
is_anomaly_detector = hasattr(loaded_model, "score_samples")

print(Fore.CYAN + Style.BRIGHT + "=" * 55)
print("        NETWORK INTRUSION DETECTION SYSTEM")
print("=" * 55 + Style.RESET_ALL)

while True:
    try:
        data, previous_network_io = collect_monitored_data(previous_network_io)
        data_buffer.append(data)

        if len(data_buffer) < SECONDS_TO_PREDICT:
            print(
                Fore.WHITE
                + "  Collecting... "
                + str(len(data_buffer))
                + "/"
                + str(SECONDS_TO_PREDICT)
                + " seconds"
                + Style.RESET_ALL,
                end="\r",
            )
            time.sleep(1)
            continue

        df = pd.DataFrame(data_buffer[-SECONDS_TO_PREDICT:])
        resampled_data = aggregate_dataframe_into_seconds_interval(
            df, SECONDS_TO_PREDICT
        )

        features = resampled_data.drop(columns=["label"])
        sample = features.iloc[[-1]]

        timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

        is_attack = False
        score_label = ""

        if is_supervised:
            probabilities = loaded_model.predict_proba(sample)[0]
            attack_prob = probabilities[1]

            if attack_prob >= PREDICTION_THRESHOLD:
                is_attack = True

            score_label = "Prob: " + format(attack_prob, ".2%")

        elif is_anomaly_detector:
            prediction = loaded_model.predict(sample)[0]
            anomaly_score = loaded_model.score_samples(sample)[0]

            if prediction == -1:
                is_attack = True

            score_label = "Score: " + format(anomaly_score, ".4f")

        else:
            prediction = loaded_model.predict(sample)[0]

            if prediction == 1:
                is_attack = True

            score_label = "Prediction: " + str(prediction)

        if is_attack:
            popup_notification(title="Attack Detected", msg=score_label)
            print(
                Fore.RED
                + Style.BRIGHT
                + "  ["
                + timestamp
                + "] ATTACK DETECTED | "
                + Style.RESET_ALL
                + Fore.RED
                + score_label
                + Style.RESET_ALL
            )
        else:
            print(
                Fore.GREEN
                + Style.BRIGHT
                + "  ["
                + timestamp
                + "] Normal Traffic  | "
                + Style.RESET_ALL
                + Fore.GREEN
                + score_label
                + Style.RESET_ALL
            )

        data_buffer = []

        time.sleep(1)

    except KeyboardInterrupt:
        print(Fore.YELLOW + "\nNIDS stopped." + Style.RESET_ALL)
        break
    except Exception:
        break